In [1]:
import numpy as np
from scipy.optimize import brentq
import scipy.integrate as scpi
from scipy.interpolate import interp1d
from scipy.integrate import trapezoid as trapz
import pandas as pd
import time
from threedellips_morison import Beam3DMatrices, AddMooringSpringsGlobal
import openturns as ot
ot.Log.Show(ot.Log.NONE)

## Part 1: Precomputation ##

Step 1.1 — Structural and geometric constants

In [2]:
E = 40e9   # Pa  (C40/50 concrete, Young's modulus)
G = 12e9   # Pa  (shear modulus)

mooring_spacing = 25.0   # [m]

mass = 266.63e3   # [kg/m]

# Ellipse outer/inner semi-axes
L_tunnel = 27000     # tunnel length [m]
z_tun    = -27.5     # tunnel centroid level [m]  (z=0 at still-water level)
ao = 14              # outer semi-major axis (horizontal) [m]
ai = 13              # inner semi-major axis [m]
bo =  8.5            # outer semi-minor axis (vertical) [m]
bi =  7.5            # inner semi-minor axis [m]

Atot = 104.6242421   # total cross-sectional area [m²] (incl. inner walls)

# Second moments of area (hollow ellipse, incl. inner walls) [m⁴]
Iy = 3256.17   # about horizontal axis  → governs vertical bending
Iz = 5950.91   # about vertical axis    → governs horizontal bending

# Polar moment for torsion
J = Iy + Iz

# Radius of gyration for mass moment of inertia [m²]
Im = ((ao*bo*(ao**2 + bo**2) - ai*bi*(ai**2 + bi**2))
      / (4*(ao*bo - ai*bi)))

Beam_EIy = E * Iy     # bending stiffness, vertical   [N·m²]
Beam_EIz = E * Iz     # bending stiffness, horizontal [N·m²]
Beam_EA  = E * Atot   # axial stiffness               [N]
Beam_GJ  = G * J      # torsional stiffness           [N·m²]

# Mooring spring stiffnesses (taut lines)
ky  = 2.03e6    # horizontal [N/m]
kz  = 10.6e6    # vertical   [N/m]
kyz = 0.102e6   # coupled    [N/m]
kzy = 0.102e6   # coupled    [N/m]\

Step 1.2 — Water properties and wave-number solver

In [3]:
rho_w = 1025.0   # water density [kg/m³]
g     =    9.81  # gravitational acceleration [m/s²]
d     =   80.0   # water depth [m]

# Morison drag coefficients (elliptical cross-section)
CD_y = 0.5   # horizontal [-]
CD_z = 1.5   # vertical   [-]


def wave_number_scalar(omega_i, depth, g=9.81):
    if omega_i <= 0:
        return 0.0
    def residual(k):
        return g*k*np.tanh(k*depth) - omega_i**2
    upper = max(10.0*omega_i**2/g + 1.0/depth, 1e-6)
    while residual(upper) < 0.0:
        upper *= 2.0
    return brentq(residual, 1e-12, upper)

Step 1.3 — Mesh

In [4]:
Le_target = 25   # target element length == mooring interval [m]

nEle_tunnel = int(round(L_tunnel / Le_target))
nNode       = nEle_tunnel + 1

TunCX = np.linspace(0, L_tunnel, nNode)
TunCY = np.zeros(nNode)
TunCZ = z_tun * np.ones(nNode)

NodeC = [[x, y, z] for x, y, z in zip(TunCX, TunCY, TunCZ)]

# NodeLeft  NodeRight  m       EA        EIy       EIz       GJ       Im
Ele = [[i, i+1, mass, Beam_EA, Beam_EIy, Beam_EIz, Beam_GJ, Im]
       for i in range(nNode - 1)]

nEle = len(Ele)
Le   = L_tunnel / nEle   # element length [m]  — used in bending recovery

print(f"Nodes: {nNode},  Elements: {nEle},  Le = {Le:.1f} m")

Nodes: 1081,  Elements: 1080,  Le = 25.0 m


Step 1.4 — Structural matrix assembly (M and K only). Ce and Qe discarderd here and assembled inside of part two of this notebook, as they are dependent on Hs and Tp.

In [5]:
LDOF = 6
nDof = LDOF * nNode

K = np.zeros((nDof, nDof))
M = np.zeros((nDof, nDof))

for iEle in range(nEle):
    n1, n2, m, EA, EIy_e, EIz_e, GJ_e, Im_e = Ele[iEle]
    n1 = int(round(n1))
    n2 = int(round(n2))

    n1dof = LDOF * n1 + np.arange(LDOF)
    n2dof = LDOF * n2 + np.arange(LDOF)

    Me, _Ce, Ke, _Qe = Beam3DMatrices(
        m         = m,
        EA        = EA,
        EIy       = EIy_e,
        EIz       = EIz_e,
        GJ        = GJ_e,
        Im        = Im_e,
        NodeCoord = [NodeC[n1], NodeC[n2]],
        ky        = ky,  kyz = kyz,  kzy = kzy,  kz = kz,
        rho_w     = rho_w,
        ao        = ao,  bo  = bo,
        CD_y      = CD_y,  CD_z = CD_z,
        a_amp     = 1.0,   # placeholder — Ce discarded
        omega     = 1.0,   # placeholder — Ce discarded
        rho_c     = 2500,
        A_tot     = Atot,
    )

    idx = np.append(n1dof, n2dof)
    for i in range(2*LDOF):
        for j in range(2*LDOF):
            M[idx[i], idx[j]] += Me[i, j]
            K[idx[i], idx[j]] += Ke[i, j]

Step 1.5 — Mooring springs

In [6]:
df_K_des = pd.read_csv("df_K_des.csv")

x_nodes    = np.array([n[0] for n in NodeC])
x_moorings = np.arange(x_nodes[0], x_nodes[-1] + mooring_spacing/2,
                       mooring_spacing)
mooring_nodes = np.array([np.argmin(np.abs(x_nodes - xm))
                           for xm in x_moorings])

K = AddMooringSpringsGlobal(K, mooring_nodes, LDOF, df_K_des)

Step 1.6 — Boundary conditions

In [7]:
NodesClamp = (0, nNode - 1)   # clamp both tunnel ends

DofsP = np.empty([0], dtype=int)
for n0 in NodesClamp:
    DofsP = np.append(DofsP, n0*LDOF + np.arange(LDOF))

DofsF  = np.setdiff1d(np.arange(nDof), DofsP)
nDofsF = len(DofsF)

print(f"Prescribed DOFs : {DofsP}")
print(f"Free DOFs       : {nDofsF}")

ff   = np.ix_(DofsF, DofsF)   # index helper — reused inside the function
M_FF = M[ff]
K_FF = K[ff]

# DOF masks — same for every (Hs, Tp), so precomputed here
mask_y = (DofsF % LDOF == 1)   
mask_z = (DofsF % LDOF == 2)   

Prescribed DOFs : [   0    1    2    3    4    5 6480 6481 6482 6483 6484 6485]
Free DOFs       : 6474


Steps 1.7 — Eigenvalue decomposition and modal basis

In [8]:
nMode                = 300    
ModalDampRatio_default = 0.05  

t_eig_start = time.time()

mat      = np.linalg.solve(M_FF, K_FF)
w2, vr   = np.linalg.eig(mat)
w_eig    = np.sqrt(np.abs(w2.real))
f_eig    = w_eig / (2.0 * np.pi)
idx_sort = f_eig.argsort()
f_eig    = f_eig[idx_sort]
vr       = vr[:, idx_sort]
PHI = vr[:, :nMode]   # (nDofsF, nMode)

Mm = np.array([PHI[:, r] @ M_FF @ PHI[:, r] for r in range(nMode)])
Km = np.array([PHI[:, r] @ K_FF @ PHI[:, r] for r in range(nMode)])

## Part 2: max_axial_stress function

In [9]:
def max_axial_stress(Hs, Tp,
                     U_c_y=0.2, U_c_z=0.0,
                     dt=0.1, t_f=None,
                     n_transient_periods=0,
                     modal_damp_ratio=ModalDampRatio_default):

    """
    Return maximum absolute axial stress in the tunnel for a regular wave.
    """


    # Step 2.1 — Wave parameters
    a_wave     = Hs / 2.0
    omega_wave = 2.0 * np.pi / Tp
    k_wave_reg = wave_number_scalar(omega_wave, d, g=g)

    C_h = np.cosh(k_wave_reg * (d + z_tun)) / np.sinh(k_wave_reg * d)
    S_v = np.sinh(k_wave_reg * (d + z_tun)) / np.sinh(k_wave_reg * d)

    a_tunnel_amp = np.sqrt(0.5 * ((a_wave * omega_wave * C_h)**2 + (a_wave * omega_wave * S_v)**2))

    # Step 2.2 — Rebuild drag and excitation matrices
    C_loc = np.zeros((nDof, nDof))
    Q_loc = np.zeros((nDof, nDof))

    for iEle in range(nEle):

        n1, n2, m, EA, EIy_e, EIz_e, GJ_e, Im_e = Ele[iEle]

        n1 = int(n1)
        n2 = int(n2)

        n1dof = LDOF * n1 + np.arange(LDOF)
        n2dof = LDOF * n2 + np.arange(LDOF)

        _, Ce, _, Qe = Beam3DMatrices(
            m         = m,
            EA        = EA,
            EIy       = EIy_e,
            EIz       = EIz_e,
            GJ        = GJ_e,
            Im        = Im_e,
            NodeCoord = [NodeC[n1], NodeC[n2]],
            ky        = ky,
            kyz       = kyz,
            kzy       = kzy,
            kz        = kz,
            rho_w     = rho_w,
            ao        = ao,
            bo        = bo,
            CD_y      = CD_y,
            CD_z      = CD_z,
            a_amp     = a_tunnel_amp,
            omega     = omega_wave,
            rho_c     = 2500,
            A_tot     = Atot,
        )

        idx = np.append(n1dof, n2dof)

        for i in range(2 * LDOF):
            for j in range(2 * LDOF):
                C_loc[idx[i], idx[j]] += Ce[i, j]
                Q_loc[idx[i], idx[j]] += Qe[i, j]

    Q_inertia = Q_loc[ff]
    Q_drag    = C_loc[ff]

    # Step 2.3 — Time vector
    if t_f is None:
        t_f = 20.0 * Tp

    tspan  = np.arange(0.0, t_f + dt, dt)
    n_time = len(tspan)

    # Step 2.4 — Wave kinematics
    theta = -omega_wave * tspan

    V_y_t    =  a_wave * omega_wave    * C_h * np.cos(theta)
    V_z_t    =  a_wave * omega_wave    * S_v * np.sin(theta)

    Vdot_y_t =  a_wave * omega_wave**2 * C_h * np.sin(theta)
    Vdot_z_t = -a_wave * omega_wave**2 * S_v * np.cos(theta)

    V_y_total = V_y_t + U_c_y
    V_z_total = V_z_t + U_c_z

    # Step 2.5 — Force vector
    F_time = np.zeros((nDofsF, n_time))

    chunk_size = 500

    for j0 in range(0, n_time, chunk_size):

        j1 = min(j0 + chunk_size, n_time)

        V_chunk = np.zeros((nDofsF, j1 - j0))
        Vdot_chunk = np.zeros((nDofsF, j1 - j0))

        V_chunk[mask_y, :] = V_y_total[j0:j1]
        V_chunk[mask_z, :] = V_z_total[j0:j1]

        Vdot_chunk[mask_y, :] = Vdot_y_t[j0:j1]
        Vdot_chunk[mask_z, :] = Vdot_z_t[j0:j1]

        F_time[:, j0:j1] = (
            Q_inertia @ Vdot_chunk
            + Q_drag @ V_chunk
        )

    # Step 2.6 — Modal solution
    Cm = 2.0 * modal_damp_ratio * np.sqrt(Mm * Km)

    Fm_time = PHI.T @ F_time

    Fm_interp = interp1d(
        tspan,
        Fm_time,
        axis=1,
        kind='linear',
        bounds_error=False,
        fill_value=(Fm_time[:, 0], Fm_time[:, -1]),
        assume_sorted=True
    )

    def qdot(t, y):

        dydt = np.empty_like(y)

        fm = Fm_interp(t)

        for r in range(nMode):

            dydt[2*r] = y[2*r + 1]

            dydt[2*r + 1] = (
                fm[r]
                - Cm[r] * y[2*r + 1]
                - Km[r] * y[2*r]
            ) / Mm[r]

        return dydt

    sol = scpi.solve_ivp(
        qdot,
        (tspan[0], tspan[-1]),
        np.zeros(2 * nMode),
        t_eval=tspan,
        method='DOP853',
        rtol=1e-6,
        atol=1e-8
    )

    # Step 2.7 — Back-project to physical DOFs
    q_modal = sol.y[0::2, :]
    x_phys = PHI @ q_modal

    q_full = np.zeros((nDof, n_time))
    q_full[DofsF, :] = x_phys

    # Step 2.8 — Element stress recovery
    n_ele = len(Ele)

    n1_arr = np.array([int(e[0]) for e in Ele])
    n2_arr = np.array([int(e[1]) for e in Ele])

    EA_arr  = np.array([e[3] for e in Ele])
    EIy_arr = np.array([e[4] for e in Ele])
    EIz_arr = np.array([e[5] for e in Ele])

    NC = np.array(NodeC)

    dX = NC[n2_arr] - NC[n1_arr]
    L_arr = np.linalg.norm(dX, axis=1)

    u1   = q_full[n1_arr * LDOF + 0, :]
    v1   = q_full[n1_arr * LDOF + 1, :]
    w1   = q_full[n1_arr * LDOF + 2, :]
    thy1 = q_full[n1_arr * LDOF + 4, :]
    thz1 = q_full[n1_arr * LDOF + 5, :]

    u2   = q_full[n2_arr * LDOF + 0, :]
    v2   = q_full[n2_arr * LDOF + 1, :]
    w2   = q_full[n2_arr * LDOF + 2, :]
    thy2 = q_full[n2_arr * LDOF + 4, :]
    thz2 = q_full[n2_arr * LDOF + 5, :]

    L    = L_arr[:, None]
    EA   = EA_arr[:, None]
    EIyv = EIy_arr[:, None]
    EIzv = EIz_arr[:, None]

    # axial force
    N = EA / L * (u2 - u1)

    # Evaluate stresses at multiple locations in every element
    x_locations = [0.0, 0.25, 0.5, 0.75, 1.0]
    sigma_global_max = 0.0

    for xbylobs in x_locations:
        Bm = np.column_stack([
            -6/L_arr**2 + 12*xbylobs/L_arr**2,
             -4/L_arr    +  6*xbylobs/L_arr,
             6/L_arr**2 - 12*xbylobs/L_arr**2,
            -2/L_arr    +  6*xbylobs/L_arr
        ])

        Bm = Bm[:, :, None]

        dof_y = np.stack(
            [v1, thz1, v2, thz2],
            axis=1
        )

        dof_z = np.stack(
            [w1, thy1, w2, thy2],
            axis=1
        )

        Mz = EIzv * np.sum(Bm * dof_y, axis=1)
        My = EIyv * np.sum(Bm * dof_z, axis=1)

        crit_pts = [
            ( 0.0,  ao),
            ( 0.0, -ao),
            ( bo,  0.0),
            (-bo,  0.0),
        ]

        sigma_all = np.empty((n_ele, 4, n_time))

        for p, (yc, zc) in enumerate(crit_pts):

            sigma_all[:, p, :] = (
                N / Atot
                + My * zc / Iy
                + Mz * yc / Iz
            )

        i_steady = int(
            np.searchsorted(
                sol.t,
                n_transient_periods * Tp
            )
        )

        sigma_here = np.max(
            np.abs(
                sigma_all[:, :, i_steady:]
            )
        )

        sigma_global_max = max(
            sigma_global_max,
            sigma_here
        )

    return float(sigma_global_max/1e6)

In [10]:
# Test
Hs_test = 2   # [m]
Tp_test = 3    # [s]

sigma = max_axial_stress(Hs_test, Tp_test)
print(f"Hs = {Hs_test} m,  Tp = {Tp_test} s  →  sigma_max = {sigma:.3f} MPa")

Hs = 2 m,  Tp = 3 s  →  sigma_max = 0.017 MPa


## Component reliability ##

In [11]:
def input_OpenTurns(X, descriptions, myLSF, failure_threshold):
    '''Set up OpenTURNs objects for MCS and FORM analysis.

    Inputs:
    X (list of ot.Distribution): List of distributions for each input variable.
    descriptions (list of str): List of descriptions for each input variable.
    myLSF (function): Limit state function.
    failure_threshold (float): Threshold for failure.

    Returns:
    None (sets global variables for OpenTURNs objects).
    '''


    global R, multinorm_copula, inputDistribution, inputRandomVector
    global myfunction, outputvector, failureevent, optimAlgo 
    global start_pt, start, algo

    R = ot.CorrelationMatrix(len(X))   
    multinorm_copula = ot.NormalCopula(R)

    inputDistribution = ot.JointDistribution(X, multinorm_copula)
    inputDistribution.setDescription(descriptions)
    inputRandomVector = ot.RandomVector(inputDistribution)

    myfunction = ot.PythonFunction(len(X), 1, myLSF)

    # Vector obtained by applying limit state function to X1 and X2
    outputvector = ot.CompositeRandomVector(myfunction, inputRandomVector)

    # Define failure event: here when the limit state function takes negative values
    failureevent = ot.ThresholdEvent(outputvector, ot.Less(), failure_threshold)
    failureevent.setName('LSF inferior to 0')

    optimAlgo = ot.Cobyla()
    optimAlgo.setMaximumCallsNumber(1000)
    optimAlgo.setMaximumAbsoluteError(1)
    optimAlgo.setMaximumRelativeError(1)
    optimAlgo.setMaximumResidualError(1)
    optimAlgo.setMaximumConstraintError(2)


def run_FORM_analysis():
    '''Run FORM analysis using OpenTurns and return key results.

    Returns:
    result (ot.FORMResult): The result of the FORM analysis.
    x_star (ot.Point): The design point in the original space.
    u_star (ot.Point): The design point in the standard normal space.
    pf (float): The probability of failure.
    beta (float): The Hasofer-Lind reliability index.
    '''
    
    start_pt = ot.Point([26.7,2.5, 4.0])

    # Start timer
    start = time.time()
    algo = ot.FORM(optimAlgo,
                   failureevent,
                   start_pt) #inputDistribution.getMean()
    algo.run()
    result = algo.getResult()
    x_star = result.getPhysicalSpaceDesignPoint()
    u_star = result.getStandardSpaceDesignPoint()
    pf = result.getEventProbability()
    beta = result.getHasoferReliabilityIndex()
    
    # End timer
    end = time.time()
    print(f'The FORM analysis took {end-start:.3f} seconds')
    
    print('FORM result, pf = {:.4f}'.format(pf))
    print('FORM result, beta = {:.3f}\n'.format(beta))
    print('The design point in the u space: ', u_star)
    print('The design point in the x space: ', x_star)
    
    return result, x_star, u_star, pf, beta    


def run_MonteCarloSimulation(mc_size):
    '''Run MCS using OpenTurns and return the probability of failure.

    Inputs:
    mc_size (int): Number of samples to generate.

    Returns:
    pf_mc (float): The probability of failure.
    '''


    # Start timer
    start = time.time()
    montecarlosize = mc_size
    outputSample = outputvector.getSample(montecarlosize)

    number_failures = sum(i < 0 for i in np.array(outputSample))[0]
    pf_mc = number_failures/montecarlosize                    

    # End timer and print the time
    end = time.time()
    print(f'The MCS took {end-start:.3f} seconds to '+
          f'evaluate {mc_size} samples.')
    
    print('pf for MCS: ', pf_mc)
    
    return pf_mc



def importance_factors(result):
    """Compute and print the importance factors using FORM results.
    
    Inputs:
    result (ot.FORMResult): The result of the FORM analysis.

    Returns:
    alpha_ot (list): Importance factors from OpenTURNs.
    alpha (list): Importance factors based on the normal vector in U-space.
    sens (list): Sensitivity of the beta to the multivariate distribution.
    """
    print(f'--- FORM Importance Factors (alpha) ---')
    import matplotlib.pyplot as plt
    plt.ion()
    print()
    alpha_ot = result.getImportanceFactors()
    print(f'\nImportance factors, from OpenTURNs:')
    [print(f'  {i:6.3f}') for i in alpha_ot]

    u_star = result.getStandardSpaceDesignPoint()
    inverseTransform = inputDistribution.getInverseIsoProbabilisticTransformation()
    failureBoundaryStandardSpace = ot.ComposedFunction(myfunction, inverseTransform)
    du0 = failureBoundaryStandardSpace.getGradient().gradient(u_star)
    g_grad = np.array(du0).transpose()[0]
    alpha = -g_grad/np.linalg.norm(g_grad)
    print('\nImportance factors, based on normal vector in U-space = ')
    [print(f'  {i:6.3f}') for i in alpha]
    print('Note: this will be different from'
          + ' result.getImportanceFactors()'
          + '\nif there are resistance variables.')

    sens = result.getHasoferReliabilityIndexSensitivity()

    print(f'\nSensitivity of Reliability Index to Multivariate Distribution')
    for i, j in enumerate(result.getHasoferReliabilityIndexSensitivity()):
        print(f'\nDistribution item number: {i}')
        print(f'  Item name: {j.getName()}')
    for k, l in zip(j.getDescription(), j):
        print(f'    {l:+6.3e} for parameter {k}')

    return alpha_ot, alpha, sens

In [12]:
# Normal(26.7,5)                                                         f_cd
# WeibullMin(beta = 0.528664, alpha = 1.2934, gamma = 0.0119857)        Hs
# LogNormal(muLog = 0.304867, sigmaLog = 0.475882, gamma = 1.4626)      Tp

X1 = ot.Normal(26.7, 5)
X2 = ot.WeibullMin(0.528664, 1.2934, 0.0119857 )
X3 = ot.LogNormal(0.304867, 0.475882, 1.4626)

X = (X1, X2, X3)
descriptions = ["f_cd", "Hs","Tp"]

failure_threshold = 0

In [13]:
def LSF(x):
    ''' 
    Vectorized limit-state function.

    Inputs: x (array of random variables)
    Output: limit-state function value
    '''

    sigma_R = float(x[0])      #float(x[0])
    Hs      = float(x[1])
    Tp      = float(x[2])

    sigma_L = max_axial_stress(Hs, Tp)

    g = sigma_R - sigma_L

    # print(f'f_cd = {x[0]:.2f},   Hs = {x[1]:.2f},   Tp={x[2]:.2f},    sigma_L = {sigma_L:.2f},    g = {g:.2f}')

    return [g]

In [14]:
input_OpenTurns(X, descriptions, LSF, failure_threshold)
result, x_star, u_star, pf_FORM, beta = run_FORM_analysis()
alpha_ot, alpha, sens = importance_factors(result)

The FORM analysis took 808.042 seconds
FORM result, pf = 0.0083
FORM result, beta = 2.396

The design point in the u space:  [-0.28508,0.641086,2.29115]
The design point in the x space:  [25.2746,0.676519,5.49831]
--- FORM Importance Factors (alpha) ---


Importance factors, from OpenTURNs:
   0.014
   0.072
   0.914

Importance factors, based on normal vector in U-space = 
  -0.100
   0.224
   0.970
Note: this will be different from result.getImportanceFactors()
if there are resistance variables.

Sensitivity of Reliability Index to Multivariate Distribution

Distribution item number: 0
  Item name: f_cd

Distribution item number: 1
  Item name: Hs

Distribution item number: 2
  Item name: Tp

Distribution item number: 3
  Item name: NormalCopula
    +0.000e+00 for parameter R_2_1_copula
    +0.000e+00 for parameter R_3_1_copula
    +0.000e+00 for parameter R_3_2_copula


In [ ]:
# pf_mc = run_MonteCarloSimulation(10000)


The MCS took 38525.235 seconds to evaluate 10000 samples.
pf for MCS:  0.0071
